# TFG V0 — 1D prototype: reversible window loading

This first version defines the basic problem on a one-dimensional grid. The register `n` stores free or occupied cells (`0 = free`, `1 = occupied`), and each index `i` represents a possible starting position for a window of size `M`.

## What the circuit does

1. Prepares a uniform superposition over the index register `idx`.
2. For each possible index `i`, reversibly copies the window `n[i:i+M]` into the work register `m`.
3. The resulting state has the conceptual form:

   ```text
   sum_i |grid>|idx=i>|window_i>
   ```

## Main registers

- `n`: classical grid encoded in qubits, with free or occupied cells.
- `idx`: index of the candidate window.
- `m`: contents of the window associated with `idx`.

## Purpose of this version

V0 is the proof of concept. Its goal is not to amplify solutions, but to verify that the mapping `idx -> window_i` works correctly and that the resulting state can be inspected with `Statevector`. Later versions reuse and generalize this reversible loading idea.

In [ ]:
import time
import qiskit
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, SparsePauliOp, partial_trace
from qiskit.visualization import plot_bloch_multivector, plot_state_qsphere
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
# DISABLED: IBM Runtime import; using simulators only. from qiskit_ibm_runtime import EstimatorV2 as Estimator
# DISABLED: IBM Runtime import; using simulators only. from qiskit_ibm_runtime import SamplerV2 as Sampler
# DISABLED: IBM Runtime import; using simulators only. from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.visualization import plot_histogram
print(qiskit.__version__)

In [ ]:
N = 4
M = 2
W = N - M + 1
k = int(np.ceil(np.log2(W))) if W > 1 else 1
print(f"Possible solutions W: {W}, qubits required to represent W positions -> k: {k}")

In [ ]:
# n: free/occupied positions
n = QuantumRegister(N, "n")
# m: work register (size M)
m = QuantumRegister(M, "m")
# idx: window selection (size k)
idx = QuantumRegister(k, "i")

qc = QuantumCircuit(n, idx, m)

# ---------------------------
# Example initial state for n
qc.x(n[2])
qc.x(n[3])
# qc.x(n[7])
# qc.x(n[13])
# ---------------------------

# Uniform superposition over indices
qc.h(idx)

# For each valid window i = 0..W-1:
# 1) apply X on idx[b] when bit b of i is 0 (control on |idx> == |i>)
# 2) copy (XOR) the window n[i:i+M] onto m[0:M]
# 3) undo the X gates
for i in range(W):
    bits = [(i >> b) & 1 for b in range(k)]  # idx[0] es LSB, idx[k-1] es MSB -> little-endian

    # Activate equality control idx == i
    for b in range(k):
        if bits[b] == 0:
            qc.x(idx[b])

    # XOR the selected window onto m
    for j in range(M):
        controls = [idx[b] for b in range(k)] + [n[i + j]]
        qc.mcx(controls, m[j])

    # Undo X gates
    for b in range(k):
        if bits[b] == 0:
            qc.x(idx[b])

qc.draw(output='mpl')

In [ ]:
sv = Statevector(qc)
sv.draw(output='latex', max_size=2**qc.num_qubits, prefix="|\\psi\\rangle = ")

In [ ]:
# Remove the n qubits (assumed to be the first N qubits: q0..q{N-1})
rho = partial_trace(sv, list(range(N)))

# Check purity: Tr(rho^2)=1 <=> pure
purity = np.real(np.trace(rho.data @ rho.data))
print(f"Purity = {purity}")

if not np.isclose(purity, 1.0, atol=1e-10):
    raise ValueError("El estado reducido no es puro; no se puede representar como un unico ket.")

# Extract the position array (n qubits) from the full state
# In Qiskit binary strings: ... n_{N-1}...n_0 (at the end)
# Display as n_0...n_{N-1} to match n[0], n[1], ...
tol_n = 1e-10
total_q = qc.num_qubits
array_posiciones = None

for basis_idx, amp in enumerate(sv.data):
    if abs(amp) <= tol_n:
        continue

    bits_full = format(basis_idx, f'0{total_q}b')
    n_desc = bits_full[-N:]
    n_asc = n_desc[::-1]

    if array_posiciones is None:
        array_posiciones = n_asc
    elif n_asc != array_posiciones:
        raise ValueError("The n qubits are not deterministic (there is no unique position array).")

if array_posiciones is None:
    raise ValueError("Could not infer the position array from the statevector.")

print(f"Position array = {array_posiciones}\n")

# Extract the ket (eigenvector with eigenvalue ~1)
vals, vecs = np.linalg.eigh(rho.data)
psi_red = Statevector(vecs[:, np.argmax(vals)])

# Display ONLY labels (without amplitudes), ordered by index
# Bit convention in psi_red: m_{M-1}...m_0 i_{k-1}...i_0
total_bits = M + k
tol = 1e-10
filas = []

for basis_idx, amp in enumerate(psi_red.data):
    if abs(amp) <= tol:
        continue

    bits = format(basis_idx, f'0{total_bits}b')
    m_desc = bits[:M]           # m_{M-1}...m_0
    i_desc = bits[M:]           # i_{k-1}...i_0

    ventana = m_desc[::-1]      # m_0...m_{M-1}
    indices = i_desc            # orden natural por indices

    filas.append((int(indices, 2), indices, ventana))

filas.sort(key=lambda x: (x[0], x[2]))

# First W results: valid windows
for _, indices, ventana in filas[:W]:
    print(f"Indices: {indices} || Window: {ventana}\n")

# Remaining results (outside the valid window range)
if len(filas) > W:
    print("----------------------------------------")
    print("Remaining states (indices outside the valid window range):\n")
    for _, indices, ventana in filas[W:]:
        print(f"Indices: {indices} || Window: {ventana}\n")